# Data préparation et nettoyage pour la calibration des sons d'écoulement urinaire

Etapes:
1. je crée mon dataset de sons de calibartion de 3 secondes et au format wav, tous associés un débit propre (target y equi repartis sur les fichiers entre 0 et 25 ml/s)
2. Vectorisation: j'extrais des features (MFCC, RMS, etc) qui décivent les caractéristiques physiques de chaque fchiers audio via le package librosa basé sur 32K pts de mesures de pressiosn pas secondes
3. je retraite certaines valeurs (log, normalisation, etc) pour que les features soient plus facilement exploitables par les futurs modèles de ML
4. Je recherche des ouliers (RobostSacaler, PCA). j'évalue la pertinence des features (matrice de correlation)
5. je crée le csv des features de calibration nettoyé (sans les outliers) et celui global pourarchivage.
6. Data enrichment: Depuis le csv nettoyé, je divise les fichiers .wav laureats en fichiers son de 0,5 secondes pour augmenter le dataset de calibration avec des features distinctes et un pas de temps similaire à celui des futures prédictions de débit en temps réel. 


In [25]:
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import operator

from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# from sklearn.ensemble import RandomForestClassifier

import glob
import os

from scipy.signal import butter, lfilter




## Feature Extraction
   Il faut caputuer des mesures physique de la pression, dela turbulence, du bruit généré, de la répartition énergétique dans le spectre, etc

améliorer la robustesse inter-téléphones en remplaçant les features absolues par des ratios et des features relatives

RMS absolu → 4 features de ratio — c'est le changement le plus important. rms_mean absolu était ta feature la plus prédictive mais aussi la plus sensible au gain. Les 4 ratios (rms_ratio_mean, rms_ratio_std, rms_peak_ratio, rms_dynamic_range) capturent la même information de débit de façon invariante au gain.
MFCC1 centré — neutralise le biais de gain sur le coefficient le plus sensible sans toucher aux autres.
S mutualisé — calculé une seule fois en haut de la fonction au lieu de 2 fois, ce qui réduit le temps de calcul.
Contraste spectral ajouté — feature naturellement insensible au gain car basée sur des différences relatives entre sous-bandes.
Attention — ces modifications changent le nombre et la nature des features (47 au lieu de 43) → il faudra réentraîner le modèle sur le dataset avec la nouvelle extract_features

In [ ]:
def extract_features_new(y, sr): # 45 features au lieu de 43
    """
    Extrait un vecteur de features acoustiques depuis un signal audio.
    
    Normalisation: Toutes les features d'énergie/amplitude sont transformées via log1p
    (ln(1+x)) avant agrégation pour réduire l'asymétrie des distributions
    et améliorer la robustesse des modèles de régression (SVR, Ridge, KNN).
    Random Forest est insensible à cette transformation mais elle ne lui nuit pas.
    
    Paramètres
    ----------
    y  : np.ndarray  — signal audio (float32, mono)
    sr : int         — fréquence d'échantillonnage (ex: 32000 Hz)
    
    Retourne
    --------
    np.ndarray de 44 features (float64)
    26 (MFCCs) + 6 (spectrales) + 2 (flatness) + 2 (RMS) +
    2 (ZCR) + 2 (flux) + 2 (ratio bande) + 1 (freq dominante) +
    2 (contraste spectral) - 1 (freq dominante scalaire) = 44
    """

    # ── Pré-calcul du spectrogramme mutualisé ─────────────────────────
    # Calculé une seule fois ici et réutilisé dans tous les blocs suivants
    # (flux, band_energy, freq_dominante) pour éviter 2 appels à stft()
    # n_fft=2048    → résolution fréquentielle ~15.6 Hz à 32000 Hz
    # hop_length=512 → frame toutes les ~16 ms
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))

    # =========================
    # 1️⃣ MFCCs — 26 features (13 mean + 13 std)
    # Capturent la "texture" globale du spectre via le cepstre mel.
    # Les 13 coefficients résument la forme de l'enveloppe spectrale :
    # MFCC1 ≈ énergie globale, MFCC2–4 ≈ forme large bande,
    # MFCC5–13 ≈ détails fins de timbre.
    # Les MFCCs peuvent être négatifs → pas de log1p applicable.
    # =========================
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=13, n_fft=2048, hop_length=512
    )
    features = []
    features.extend(np.mean(mfcc, axis=1))  # 13 valeurs
    features.extend(np.std(mfcc, axis=1))   # 13 valeurs

    # =========================
    # 2️⃣ Features spectrales — 6 features (3 x mean+std)
    # Toutes en Hz → valeurs larges et asymétriques → log1p appliqué
    # sur le vecteur de frames AVANT agrégation.
    # Centroïde, bandwidth, rolloff sont des ratios d'énergie fréquentielle
    # → naturellement peu sensibles au gain absolu ✅
    # =========================
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_centroid = np.log1p(spec_centroid)
    features.append(np.mean(spec_centroid))
    features.append(np.std(spec_centroid))

    spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spec_bandwidth = np.log1p(spec_bandwidth)
    features.append(np.mean(spec_bandwidth))
    features.append(np.std(spec_bandwidth))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)
    rolloff = np.log1p(rolloff)
    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    # =========================
    # 3️⃣ Spectral flatness — 2 features (mean + std)
    # Borné entre 0 et 1 → pas de log1p.
    # Ratio géométrique/arithmétique → insensible au gain ✅
    # =========================
    spec_flatness = librosa.feature.spectral_flatness(y=y)
    features.append(np.mean(spec_flatness))
    features.append(np.std(spec_flatness))

    # =========================
    # 4️⃣ RMS Energy — 2 features (mean + std)
    # Feature la plus directement corrélée au débit.
    # log1p appliqué car valeurs très asymétriques.
    # ⚠️ Sensible au gain du micro — à améliorer si dataset
    #    multi-téléphones disponible (remplacer par ratios)
    # =========================
    rms = librosa.feature.rms(y=y)
    rms = np.log1p(rms)
    features.append(np.mean(rms))
    features.append(np.std(rms))

    # =========================
    # 5️⃣ Zero-Crossing Rate — 2 features (mean + std)
    # Borné entre 0 et 1 → pas de log1p.
    # =========================
    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    # =========================
    # 6️⃣ Flux spectral — 2 features (mean + std)
    # Normalisé par S.mean() * S.shape[0] → insensible au gain ✅
    # Réutilise S déjà calculé en haut de la fonction.
    # log1p car valeurs positives et asymétriques.
    # =========================
    flux = np.sqrt(np.sum(np.diff(S, axis=1)**2, axis=0)) / (S.mean() * S.shape[0])
    flux = np.log1p(flux)
    features.append(np.mean(flux))
    features.append(np.std(flux))

    # =========================
    # 7️⃣ Ratio énergie bande 1–4 kHz — 2 features (mean + std)
    # band_energy / total_energy → insensible au gain ✅
    # Réutilise S déjà calculé en haut de la fonction.
    # =========================
    freqs        = librosa.fft_frequencies(sr=sr)
    band         = np.logical_and(freqs >= 1000, freqs <= 4000)
    band_energy  = (S[band, :] ** 2).mean(axis=0)
    total_energy = (S ** 2).mean(axis=0) + 1e-8
    ratio_band_energy = band_energy / total_energy
    features.append(np.mean(ratio_band_energy))
    features.append(np.std(ratio_band_energy))

    # =========================
    # 8️⃣ Fréquence dominante normalisée — 1 feature
    # Normalisée par Nyquist (sr/2) → entre 0 et 1 ✅
    # Réutilise S et freqs déjà calculés.
    # =========================
    freq_dominante      = freqs[np.argmax(S.mean(axis=1))]
    freq_dominante_norm = freq_dominante / (sr / 2)
    features.append(float(freq_dominante_norm))

    # =========================
    # 9️⃣ Contraste spectral — 2 features (mean + std)
    # Différence entre pics et vallées du spectre par sous-bande.
    # Feature de FORME spectrale → insensible au gain absolu ✅
    # n_bands=4 → découpe le spectre en 4 sous-bandes
    # Un jet puissant génère des pics d'énergie marqués dans 1-4 kHz
    # → contraste élevé, discriminant entre débits faible et fort.
    # =========================
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=4)
    features.append(float(np.mean(contrast)))
    features.append(float(np.std(contrast)))

    # Vecteur final :
    # 26 (MFCCs) + 6 (spectrales) + 2 (flatness) + 2 (RMS) +
    # 2 (ZCR) + 2 (flux) + 2 (ratio bande) + 1 (freq dominante) +
    # 2 (contraste) = 45 features
    return np.array(features)

In [1]:
def extract_features(y, sr):
    """
    Extrait un vecteur de features acoustiques depuis un signal audio.
    
    Normalsiation: Toutes les features d'énergie/amplitude sont transformées via log1p
    (ln(1+x)) avant agrégation pour réduire l'asymétrie des distributions
    et améliorer la robustesse des modèles de régression (SVR, Ridge, KNN).
    Random Forest est insensible à cette transformation mais elle ne lui nuit pas.
    
    Paramètres
    ----------
    y  : np.ndarray  — signal audio (float32, mono)
    sr : int         — fréquence d'échantillonnage (ex: 44100 Hz)
    
    Retourne
    --------
    np.ndarray de 42 features (float64)
    """

    # -------------------------
    # 0️⃣ Pré-traitement robuste micro
    # -------------------------
    
    # Filtre passe-haut du premier ordre (coeff ~0.97) qui atténue les très basses
    # fréquences (<80 Hz) : bruits de ventilation, vibrations de la pièce, grondements.
    # Désactivé ici car son effet sur la bande 1–4 kHz est négligeable,
    # et il peut légèrement déformer les MFCCs bas (coeff 1–3).
    # À réactiver si le bruit de fond basse fréquence est un problème constaté.
    # y = librosa.effects.preemphasis(y)

    # =========================
    # 1️⃣ MFCCs — 26 features (13 mean + 13 std)
    # Capturent la "texture" globale du spectre via le cepstre mel.
    # Les 13 coefficients résument la forme de l'enveloppe spectrale :
    # MFCC1 ≈ énergie globale, MFCC2–4 ≈ forme large bande,
    # MFCC5–13 ≈ détails fins de timbre.
    # n_fft=2048 → résolution fréquentielle de ~21 Hz à 44100 Hz
    # hop_length=512 → frame toutes les ~11.6 ms
    # =========================
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13,
        n_fft=2048,
        hop_length=512
    )

    # CMVN (Cepstral Mean and Variance Normalization) : soustrait la moyenne
    # par coefficient pour neutraliser les effets de la réponse en fréquence
    # du microphone. Utile si les enregistrements proviennent de téléphones
    # très différents. Désactivé car peut effacer des informations utiles
    # sur le débit si les MFCCs ont une moyenne globalement corrélée au débit.
    # mfcc = (mfcc - np.mean(mfcc, axis=1, keepdims=True)) / \
    #        (np.std(mfcc, axis=1, keepdims=True) + 1e-8)

    # Version allégée : soustraction de la moyenne uniquement (sans normalisation
    # de la variance). Compromis entre robustesse inter-micros et conservation
    # de l'information de débit. Également désactivé pour les mêmes raisons.
    # mfcc = mfcc - np.mean(mfcc, axis=1, keepdims=True)

    # Les MFCCs peuvent être négatifs → pas de log1p applicable.
    # On agrège directement par mean et std sur l'axe temporel (axis=1).
    features = []
    features.extend(np.mean(mfcc, axis=1))  # 13 valeurs
    features.extend(np.std(mfcc, axis=1))   # 13 valeurs

    # =========================
    # 2️⃣ Features spectrales — 6 features (3 x mean+std)
    # Décrivent la distribution de l'énergie dans le domaine fréquentiel.
    # Toutes en Hz → valeurs larges et asymétriques → log1p appliqué
    # sur le vecteur de frames AVANT agrégation (mean/std cohérentes).
    # =========================

    # Centroïde spectral : "centre de gravité" du spectre en Hz.
    # Monte avec le débit car un jet puissant génère plus d'énergie
    # dans les hautes fréquences.
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_centroid = np.log1p(spec_centroid)
    features.append(np.mean(spec_centroid))
    features.append(np.std(spec_centroid))

    # Bandwidth spectrale : étalement du spectre autour du centroïde en Hz.
    # Un impact fort génère un spectre plus large → bandwidth élevée.
    spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spec_bandwidth = np.log1p(spec_bandwidth)
    features.append(np.mean(spec_bandwidth))
    features.append(np.std(spec_bandwidth))

    # Rolloff spectral : fréquence en Hz en dessous de laquelle se concentre
    # 85% de l'énergie. Discrimine les débits faibles (rolloff bas, énergie
    # concentrée dans les graves) des débits forts (rolloff élevé).
    # roll_percent=0.85 est le seuil standard, ajustable à 0.90 si besoin.
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)
    rolloff = np.log1p(rolloff)
    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    # =========================
    # 3️⃣ Spectral flatness — 2 features (mean + std)
    # Rapport géométrique/arithmétique du spectre, borné entre 0 et 1.
    # 0 → signal tonal (énergie concentrée sur quelques fréquences)
    # 1 → bruit blanc (énergie uniformément répartie)
    # Le bruit d'impact du jet est large bande → flatness élevée.
    # Utile pour détecter et distinguer le son du jet du bruit de fond ambiant
    # (voix, ventilation) qui aura une flatness différente et plus stable.
    # Pas de log1p car déjà borné entre 0 et 1 → pas d'asymétrie problématique.
    # =========================
    spec_flatness = librosa.feature.spectral_flatness(y=y)
    features.append(np.mean(spec_flatness))
    features.append(np.std(spec_flatness))

    # =========================
    # 4️⃣ RMS Energy — 2 features (mean + std)
    # Mesure la puissance moyenne du signal par frame.
    # Feature la plus directement corrélée au débit : un jet fort
    # génère un impact sonore plus puissant → RMS plus élevé.
    # log1p appliqué car valeurs très proches de 0 et très asymétriques
    # (silences avant/après miction tirent la distribution vers le bas).
    # =========================
    rms = librosa.feature.rms(y=y)
    rms = np.log1p(rms)
    features.append(np.mean(rms))
    features.append(np.std(rms))

    # =========================
    # 5️⃣ Zero-Crossing Rate — 2 features (mean + std)
    # Nombre de fois que le signal change de signe par frame, normalisé.
    # Proxy de la rugosité temporelle et de la richesse en hautes fréquences.
    # Utile pour distinguer le silence (ZCR faible et stable) du son d'impact
    # (ZCR plus élevé et variable). Borné entre 0 et 1 → pas de log1p.
    # =========================
    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    # =========================
    # 6️⃣ Flux spectral — 2 features (mean + std)
    # Mesure la variation du spectre entre frames successives.
    # Un jet turbulent (débit élevé) génère un spectre instable → flux élevé.
    # Un jet laminaire (débit faible) a un spectre stable → flux bas.
    # Feature dynamique complémentaire aux features statiques ci-dessus.
    # Normalisé par S.mean() * S.shape[0] pour être indépendant du volume
    # global → robustesse inter-téléphones.
    # log1p car valeurs positives et asymétriques.
    # Même n_fft/hop_length que les MFCCs pour cohérence temporelle.
    # =========================
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))# S= spectrogramme de magnitude de la frame. STFT = Short-Time Fourier Transform
    flux = np.sqrt(np.sum(np.diff(S, axis=1)**2, axis=0)) / (S.mean() * S.shape[0])
    flux = np.log1p(flux)
    features.append(np.mean(flux))
    features.append(np.std(flux))

    # =========================
    # 7️⃣ Énergie bande 1–4 kHz — 2 features (mean + std)
    # Zone fréquentielle la plus informative pour le bruit d'impact
    # d'un jet liquide sur l'eau (documenté en acoustique des jets).
    # Avantage clé : les microphones de smartphones sont plus homogènes
    # dans cette bande qu'en dehors → robustesse inter-appareils.
    # Calculé sur les puissances (S²) plutôt que les amplitudes (S)
    # pour une mesure d'énergie physiquement correcte.
    # log1p car valeurs de puissance très asymétriques ou Alternative normalisée (ratio énergie bande / énergie totale)
    # =========================
    # S = np.abs(librosa.stft(y))  # ⚠️ identique au S du bloc 6️⃣ → peut être mutualisé
    freqs = librosa.fft_frequencies(sr=sr)
    band  = np.logical_and(freqs >= 1000, freqs <= 4000)

    # Énergie dans la bande 1-4 kHz (puissance par frame)
    band_energy  = (S[band, :] ** 2).mean(axis=0)      # shape (n_frames,)

    # Énergie totale par frame
    total_energy = (S ** 2).mean(axis=0) + 1e-8        # shape (n_frames,)

    # Ratio : énergie bande / énergie totale → entre 0 et 1
    ratio_band_energy = band_energy / total_energy      # shape (n_frames,)

    features.append(np.mean(ratio_band_energy))
    features.append(np.std(ratio_band_energy))

    
    # Fréquence dominante = fréquence avec l'énergie maximale de la frame
    freqs          = librosa.fft_frequencies(sr=sr)
    freq_dominante = freqs[np.argmax(S.mean(axis=1))]  # scalaire unique en Hz

    # Normalisation : ramène la fréquence dominante entre 0 et 1
    # en divisant par la fréquence de Nyquist (sr/2)
    freq_dominante_norm = freq_dominante / (sr / 2)

    features.append(float(freq_dominante_norm))


    # Vecteur final : 26 (MFCCs) + 6 (spectrales) + 2 (flatness) +
    #                  2 (RMS) + 2 (ZCR) + 2 (flux) + 2 (bande) = 42 features
    return np.array(features)

# Retraitement, normalisation des features (Optionnel, à tester)

In [27]:
def preprocess_audio(
    file,                     # chemin du fichier audio
    sr=16000,                 # fréquence cible après resampling.Tester 16000 (téléphone) plutpot que 32000?
    lowcut=150,               # fréquence minimale utile pour le flux liquide
    highcut=8000,             # fréquence maximale utile
    filter_order=4,           # ordre du filtre passe-bande
    target_rms=0.1,           # niveau RMS cible (stabilise les micros)
    compression_strength=0.7, # compression dynamique (réduit écarts entre appareils)
    trim_signal=False,         # active ou non la suppression du silence début/fin
    trim_padding=1.0,         # durée conservée avant/après le flux détecté (secondes)
    max_internal_silence=2.0  # silence max autorisé au milieu d'un flux (secondes)
):
    """
    Prétraitement complet pour analyse acoustique du débit urinaire.
    """

    # ---------------------------------------------------------
    # 1) Chargement + mono + resampling
    # ---------------------------------------------------------
    y, sr = librosa.load(file, sr=sr, mono=True)

    # ---------------------------------------------------------
    # 2) Réduction du bruit ambiant (spectral gating)
    # ---------------------------------------------------------
    stft = librosa.stft(y)
    magnitude, phase = librosa.magphase(stft)

    noise_profile = np.mean(magnitude[:, :20], axis=1, keepdims=True)
    reduction_factor = 1.5
    cleaned_mag = np.maximum(magnitude - reduction_factor * noise_profile, 0)

    y = librosa.istft(cleaned_mag * phase)

    # ---------------------------------------------------------
    # 3) Suppression silence début / fin (optionnelle)
    # avec gestion des pauses internes
    # ---------------------------------------------------------
    if trim_signal:

        rms = librosa.feature.rms(y=y)[0]
        threshold = np.percentile(rms, 65)

        frame_length = int(len(y) / len(rms))
        frame_time = frame_length / sr

        active = rms > threshold
        active_idx = np.where(active)[0]

        if len(active_idx) > 0:

            segments = []
            start = active_idx[0]

            for i in range(1, len(active_idx)):
                gap = active_idx[i] - active_idx[i-1]

                silence_duration = gap * frame_time

                if silence_duration > max_internal_silence:
                    segments.append((start, active_idx[i-1]))
                    start = active_idx[i]

            segments.append((start, active_idx[-1]))

            global_start = segments[0][0]
            global_end = segments[-1][1]

            start_sample = max(0, int(global_start * frame_length - trim_padding * sr))
            end_sample = min(len(y), int(global_end * frame_length + trim_padding * sr))

            y = y[start_sample:end_sample]

    # ---------------------------------------------------------
    # 4) Suppression offset DC
    # ---------------------------------------------------------
    y = y - np.mean(y)

    # ---------------------------------------------------------
    # 5) Normalisation RMS
    # ---------------------------------------------------------
    rms = np.sqrt(np.mean(y**2) + 1e-9)
    y = y * (target_rms / rms)

    # ---------------------------------------------------------
    # 6) Filtre turbulence liquide
    # ---------------------------------------------------------
    nyq = 0.5 * sr
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(filter_order, [low, high], btype="band")
    y = lfilter(b, a, y)

    # ---------------------------------------------------------
    # 7) Compression dynamique
    # ---------------------------------------------------------
    y = np.sign(y) * (np.abs(y) ** compression_strength)

    # ---------------------------------------------------------
    # 8) Normalisation finale: suppression de l'information lié au materiel
    # ---------------------------------------------------------

    '''suppression essentiellement :

            volume global du micro
            distance au micro
            gain automatique
            différences de niveau entre appareils
    '''
    y = librosa.util.normalize(y)

    return y, sr


## Build Dataset of extracted features (Calibration or Test)

In [ ]:
def build_dataset(folder='calibration_sounds_features', margin_sec=3):
    rows = []

    feature_names = [
        "mfcc1_mean","mfcc2_mean","mfcc3_mean","mfcc4_mean","mfcc5_mean",
        "mfcc6_mean","mfcc7_mean","mfcc8_mean","mfcc9_mean","mfcc10_mean",
        "mfcc11_mean","mfcc12_mean","mfcc13_mean",

        "mfcc1_std","mfcc2_std","mfcc3_std","mfcc4_std","mfcc5_std",
        "mfcc6_std","mfcc7_std","mfcc8_std","mfcc9_std","mfcc10_std",
        "mfcc11_std","mfcc12_std","mfcc13_std",

        "spec_centroid_mean","spec_centroid_std",
        "spec_band_mean","spec_band_std",
        "spec_rolloff_mean","spec_rolloff_std",
        "spec_flat_mean","spec_flat_std",
        "rms_mean","rms_std",
        "zero_cross_rate_mean","zero_cross_rate_std",
        "flux_mean","flux_std",
        "band_energy_mean","band_energy_std","freq_dominante_norm"
        "debit"
    ]

    files = glob.glob(os.path.join(folder, "*.wav"))

    files_sorted = sorted(
        files,
        key=lambda x: float(os.path.basename(x).split("_")[0])
    )

    for file in files_sorted:

        debit = float(os.path.basename(file).partition('_')[0])

        y, sr = librosa.load(file, sr=32000) # potentiellment remettre à 16KHz
        # Normalisation du son avant extraction des features
        # y, sr = preprocess_audio(file, sr=32000, trim_signal=False, trim_padding=margin_sec)# potentiellment remettre à 32KHz

        # print(" vecteur y initial")
        # print (y)
        # print("shape:", y.shape)
        

        feat = extract_features(y, sr)
        rows.append(np.append(feat, debit))

    df = pd.DataFrame(rows,columns=feature_names)
    df.to_csv(folder + "_features.csv", index=False)
    print(f'Dataset saved as {folder}_features.csv')



# Build Calibration Dataset 

In [29]:

build_dataset('sons_tests')

df_test=pd.read_csv("sons_tests_features.csv", sep=",")

Dataset saved as sons_tests_features.csv


En plus des données de test issues du dataset global, un jeu d'autres mesures avec des micros tiers a été réalisé.

In [30]:
build_dataset('calibration_sounds')

df=pd.read_csv("calibration_sounds_features.csv", sep=",")

Dataset saved as calibration_sounds_features.csv


In [7]:
df.columns

Index(['mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean',
       'mfcc6_mean', 'mfcc7_mean', 'mfcc8_mean', 'mfcc9_mean', 'mfcc10_mean',
       'mfcc11_mean', 'mfcc12_mean', 'mfcc13_mean', 'mfcc1_std', 'mfcc2_std',
       'mfcc3_std', 'mfcc4_std', 'mfcc5_std', 'mfcc6_std', 'mfcc7_std',
       'mfcc8_std', 'mfcc9_std', 'mfcc10_std', 'mfcc11_std', 'mfcc12_std',
       'mfcc13_std', 'spec_centroid_mean', 'spec_centroid_std',
       'spec_band_mean', 'spec_band_std', 'spec_rolloff_mean',
       'spec_rolloff_std', 'spec_flat_mean', 'spec_flat_std', 'rms_mean',
       'rms_std', 'zero_cross_rate_mean', 'zero_cross_rate_std', 'flux_mean',
       'flux_std', 'band_energy_mean', 'band_energy_std', 'debit'],
      dtype='object')

# Features Analysis

In [37]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EllipticEnvelope
from scipy import stats

def detect_outliers_par_debit(df, features_cols, debit_col='debit',
                               n_bins=10, methode='zscore',
                               contamination=0.1, seuil_zscore=2.5):

    df = df.copy()
    df['outlier']   = False
    df['debit_bin'] = pd.cut(df[debit_col], bins=n_bins)

    # ── 1. Détection par plage ────────────────────────────────────────
    for bin_val, groupe in df.groupby('debit_bin', observed=True):
        idx = groupe.index
        X   = groupe[features_cols].values

        if len(X) < 3:
            continue

        if methode == 'zscore':
            z            = np.abs(stats.zscore(X, axis=0))
            outlier_mask = (z > seuil_zscore).any(axis=1)
            df.loc[idx, 'outlier'] = outlier_mask

        elif methode == 'mahalanobis':
            if len(X) < len(features_cols) + 2:
                z            = np.abs(stats.zscore(X, axis=0))
                outlier_mask = (z > seuil_zscore).any(axis=1)
                df.loc[idx, 'outlier'] = outlier_mask
                continue
            try:
                detector     = EllipticEnvelope(contamination=contamination,
                                                random_state=42)
                detector.fit(X)
                predictions  = detector.predict(X)
                df.loc[idx, 'outlier'] = (predictions == -1)
            except Exception:
                z            = np.abs(stats.zscore(X, axis=0))
                outlier_mask = (z > seuil_zscore).any(axis=1)
                df.loc[idx, 'outlier'] = outlier_mask

    # ── 2. Scaling + PCA ─────────────────────────────────────────────
    scaler   = RobustScaler()
    X_scaled = scaler.fit_transform(df[features_cols].values)

    pca  = PCA(n_components=2)
    X_2d = pca.fit_transform(X_scaled)

    var1, var2 = pca.explained_variance_ratio_ * 100
    print(f"Variance expliquée — PC1: {var1:.1f}%  PC2: {var2:.1f}%  "
          f"Total: {var1+var2:.1f}%")
    print(f"Outliers détectés : {df['outlier'].sum()} / {len(df)}")

    # ── 3. Graphique Plotly interactif ────────────────────────────────
    outliers  = df['outlier'].values
    debits    = df[debit_col].values
    bins      = df['debit_bin'].astype(str).values
    indices   = df.index.values

    bin_labels = df['debit_bin'].cat.categories.astype(str).tolist()
    n_bins_eff = len(bin_labels)

    # Camaïeu bleu : très clair (faible débit) → très foncé (fort débit)
    # Généré programmatiquement pour s'adapter à n_bins quelconque
    # On évite le blanc pur (illisible sur fond sombre) → on part de ~85% luminosité
    def blue_camaieu(n):
        """
        Retourne n couleurs en camaïeu bleu du plus clair au plus foncé.
        Clair = faible débit, foncé = fort débit.
        """
        colors = []
        for i in range(n):
            # t va de 0 (clair) à 1 (foncé)
            t = i / max(n - 1, 1)
            # RGB interpolé :
            # clair  : (180, 210, 255)  → bleu très pâle presque blanc
            # foncé  : (0,   40,  120)  → bleu marine profond
            r = int(180 - t * 180)
            g = int(210 - t * 170)
            b = int(255 - t * 135)
            colors.append(f'rgb({r},{g},{b})')
        return colors

    palette       = blue_camaieu(n_bins_eff)
    bin_color_map = {label: palette[i] for i, label in enumerate(bin_labels)}

    fig = go.Figure()

    # ── Points normaux : une trace par bin ───────────────────────────
    for i, bin_label in enumerate(bin_labels):
        mask = (~outliers) & (bins == bin_label)
        if not mask.any():
            continue

        hover_text = [
            f"<b>Index :</b> {indices[j]}<br>"
            f"<b>Débit :</b> {debits[j]:.3f} mL/s<br>"
            f"<b>Bin :</b> {bins[j]}<br>"
            f"<b>PC1 :</b> {X_2d[j, 0]:.3f}<br>"
            f"<b>PC2 :</b> {X_2d[j, 1]:.3f}"
            for j in np.where(mask)[0]
        ]

        fig.add_trace(go.Scatter(
            x         = X_2d[mask, 0],
            y         = X_2d[mask, 1],
            mode      = 'markers',
            name      = bin_label,
            marker    = dict(
                color   = bin_color_map[bin_label],
                size    = 9,
                opacity = 0.90,
                line    = dict(width=0.8, color='rgba(255,255,255,0.3)')
            ),
            text      = hover_text,
            hoverinfo = 'text'
        ))

    # ── Outliers : croix rouges vives ────────────────────────────────
    if outliers.any():
        hover_outliers = [
            f"<b>⚠️ OUTLIER</b><br>"
            f"<b>Index :</b> {indices[j]}<br>"
            f"<b>Débit :</b> {debits[j]:.3f} mL/s<br>"
            f"<b>Bin :</b> {bins[j]}<br>"
            f"<b>PC1 :</b> {X_2d[j, 0]:.3f}<br>"
            f"<b>PC2 :</b> {X_2d[j, 1]:.3f}"
            for j in np.where(outliers)[0]
        ]

        fig.add_trace(go.Scatter(
            x         = X_2d[outliers, 0],
            y         = X_2d[outliers, 1],
            mode      = 'markers',
            name      = '⚠️ Outliers',
            marker    = dict(
                symbol  = 'x',
                size    = 14,
                color   = '#ff3333',
                opacity = 1.0,
                line    = dict(width=2.5, color='#ff3333')
            ),
            text      = hover_outliers,
            hoverinfo = 'text'
        ))

    fig.update_layout(
        title        = (f"PCA — Détection outliers ({methode}) | "
                        f"{df['outlier'].sum()} outliers / {len(df)} samples"),
        xaxis_title  = f"PC1 ({var1:.1f}% variance)",
        yaxis_title  = f"PC2 ({var2:.1f}% variance)",
        legend_title = "Bin débit (mL/s)<br><sup>clair=faible → foncé=fort</sup>",
        hovermode    = 'closest',
        template     = 'plotly_dark',
        width        = 950,
        height       = 650,
        legend       = dict(
            bgcolor     = 'rgba(0,0,0,0.4)',
            bordercolor = 'rgba(255,255,255,0.2)',
            borderwidth = 1,
            traceorder  = 'normal'
        )
    )

    fig.show()

    # ── 4. Retour ─────────────────────────────────────────────────────
    df_clean = df[~df['outlier']].drop(columns=['debit_bin']).reset_index(drop=True)
    return df, df_clean

In [38]:
features_cols = [
    # Énergie — feature la plus prédictive du débit
    'rms_mean',
    'rms_std',           # important : capture les variations d'intensité du jet

    # Bande 1–4 kHz — ta feature la plus robuste inter-téléphones
    'band_energy_mean',
    'band_energy_std',

    # Turbulence / dynamique temporelle
    'flux_mean',         # corrélé aux turbulences du jet

    # Position spectrale — un seul représentant suffit
    'spec_centroid_mean',

    # Caractère tonal vs bruité — discrimine jet vs bruit de fond
    'spec_flat_mean',

    # MFCCs — garder seulement les premiers (basse fréquence cepstrale)
    # MFCC1 ≈ énergie globale, MFCC2-4 ≈ forme large bande
    # Au-delà de MFCC5 on capture des détails fins peu liés au débit
    'mfcc1_mean',
    'mfcc2_mean',
    'mfcc3_mean',
    'mfcc4_mean',
]

In [39]:

# features_cols = [
#     'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean',
#     'mfcc5_mean', 'mfcc6_mean'
# ]

#toutes les colonnes :
# features_cols = [col for col in df.columns if col != 'debit']

# Option 4 : toutes les colonnes numériques sauf debit
# features_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# features_cols.remove('debit')

# ── Usage ─────────────────────────────────────────────────────────────
df_annotated, df_clean = detect_outliers_par_debit(
    df,
    features_cols = features_cols,
    n_bins        = 10,
    methode       = 'zscore',        # 'zscore' ou 'mahalanobis' avec contanimation = 0.15,  # force 15% d'outliers minimum
    seuil_zscore  = 2
)

# Inspecter manuellement avant de valider
print(df_annotated[df_annotated['outlier']][['debit'] + features_cols])

# Puis découper en frames seulement sur df_clean

Variance expliquée — PC1: 55.3%  PC2: 23.8%  Total: 79.2%
Outliers détectés : 26 / 143


      debit  rms_mean   rms_std  band_energy_mean  band_energy_std  flux_mean  \
6     0.010  0.002199  0.000837          0.001324         0.000798   0.041671   
14    4.362  0.028688  0.013314          0.910318         0.710316   0.044582   
17    5.000  0.034329  0.012070          1.217915         0.636413   0.048622   
20    5.280  0.002568  0.001440          0.002372         0.002480   0.057086   
24    5.777  0.031335  0.020705          0.724383         0.568205   0.046754   
27    6.250  0.040114  0.013291          1.273120         0.649136   0.045856   
28    6.307  0.002404  0.001578          0.003028         0.004223   0.052836   
39    7.861  0.003614  0.001407          0.003939         0.003460   0.046693   
41    7.898  0.003146  0.001464          0.003981         0.004634   0.048024   
43    8.198  0.003143  0.001122          0.002921         0.002657   0.050506   
54    9.099  0.047220  0.017639          1.384519         0.674595   0.047900   
65   10.660  0.043148  0.005

In [40]:
df_annotated

,mfcc1_mean,mfcc2_mean,mfcc3_mean,mfcc4_mean,mfcc5_mean,mfcc6_mean,mfcc7_mean,mfcc8_mean,mfcc9_mean,mfcc10_mean,...,rms_std,zero_cross_rate_mean,zero_cross_rate_std,flux_mean,flux_std,band_energy_mean,band_energy_std,debit,outlier,debit_bin
0,-716.933533,50.742298,17.720356,-4.217847,29.864586,-10.154039,23.325575,-7.101023,7.525615,3.602954,...,0.000009,0.461784,0.038070,0.023080,0.001959,0.000008,0.000001,0.000,False,"(-0.0254, 2.544]"
1,-692.237793,53.579861,6.129889,12.243140,30.468887,-3.181616,29.806770,-12.598526,10.581248,8.964967,...,0.000032,0.283642,0.051863,0.026538,0.006048,0.000025,0.000036,0.001,False,"(-0.0254, 2.544]"
2,-493.638885,61.116085,24.674517,13.870263,13.879618,10.880253,7.687804,9.044422,7.082927,8.582839,...,0.000482,0.109762,0.031946,0.045611,0.018723,0.000625,0.000306,0.002,False,"(-0.0254, 2.544]"
3,-607.613708,59.203156,27.292942,12.623013,13.818491,7.761862,5.962965,5.791190,4.362058,6.157206,...,0.000240,0.133670,0.038757,0.041929,0.025501,0.000070,0.000076,0.003,False,"(-0.0254, 2.544]"
4,-805.906921,90.039322,12.957466,27.510977,16.195431,12.622943,13.311118,7.969684,10.154236,5.398314,...,0.000055,0.106217,0.054036,0.051397,0.019333,0.000002,0.000003,0.005,False,"(-0.0254, 2.544]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,-100.427010,85.814667,-41.768116,1.733437,-22.751984,-17.328894,14.527456,-19.693165,-10.299988,-10.127850,...,0.015187,0.128685,0.023191,0.032460,0.008024,2.319635,0.449503,23.148,False,"(22.9, 25.445]"
139,-108.642776,82.750664,-43.195236,0.970258,-16.618874,-24.157408,14.797751,-23.555513,-9.403457,-14.069793,...,0.014799,0.122431,0.024616,0.032784,0.007373,2.152692,0.374084,23.255,False,"(22.9, 25.445]"
140,-265.617157,44.329910,-0.402523,49.045994,-47.854752,18.409088,-24.881561,15.967571,-17.192636,3.983510,...,0.004011,0.203821,0.052102,0.027446,0.005981,0.079836,0.032880,25.000,False,"(22.9, 25.445]"
141,-90.534477,78.470352,-44.844112,-1.934222,-18.844818,-17.063074,11.050971,-21.422199,-11.050311,-12.393999,...,0.016656,0.134316,0.031853,0.030870,0.007391,2.570048,0.477495,25.380,False,"(22.9, 25.445]"


In [41]:
import ipywidgets as widgets
from IPython.display import display

outliers_df = df_annotated[df_annotated['outlier']]
clean_df    = df_annotated[~df_annotated['outlier']]

print("=== Débits des outliers ===")
print(outliers_df['debit'].describe())
print("\n=== Débits des points OK ===")
print(clean_df['debit'].describe())

print("\n=== Outliers par bin de débit ===")
print(outliers_df['debit_bin'].value_counts().sort_index())

print('----------------------------------------------------------------------')
print("Outlier coché = outlier à supprimer")
print('----------------------------------------------------------------------')



=== Débits des outliers ===
count    26.000000
mean     10.751962
std       4.932990
min       0.010000
25%       6.695500
50%      11.055500
75%      14.367500
max      19.762000
Name: debit, dtype: float64

=== Débits des points OK ===
count    117.000000
mean      11.240983
std        5.966384
min        0.000000
25%        7.248000
50%       10.972000
75%       14.771000
max       25.445000
Name: debit, dtype: float64

=== Outliers par bin de débit ===
debit_bin
(-0.0254, 2.544]    1
(2.544, 5.089]      2
(5.089, 7.634]      4
(7.634, 10.178]     4
(10.178, 12.722]    6
(12.722, 15.267]    5
(15.267, 17.812]    2
(17.812, 20.356]    2
(20.356, 22.9]      0
(22.9, 25.445]      0
Name: count, dtype: int64
----------------------------------------------------------------------
Outlier coché = outlier à supprimer
----------------------------------------------------------------------


In [42]:
# Affichage interactif pour validation manuelle
to_remove = set()

def make_observer(idx):
    """
    Crée une fonction d'observation propre pour chaque idx.
    
    """
    def observer(change):
        if change['new']:
            to_remove.add(idx)
            print(f"  ➕ [{idx}] marqué pour suppression  | to_remove={sorted(to_remove)}")
        else:
            to_remove.discard(idx)
            print(f"  ✅ [{idx}] conservé                 | to_remove={sorted(to_remove)}")
    return observer

for idx in outliers_df.index:
    row = df.loc[idx]

    # Ajoute directement dans to_remove car coché par défaut
    to_remove.add(idx)

    label_text = (
        f"[{idx}]  "
        f"débit={row['debit']:.3f} mL/s  |  "
        f"rms={row['rms_mean']:.4f}  |  "
        f"flux={row['flux_mean']:.4f}  |  "
        f"centroid={row['spec_centroid_mean']:.1f}"
    )

    cb = widgets.Checkbox(
        value=True,
        description='',
        layout=widgets.Layout(width='30px'),
        indent=False
    )
    label = widgets.Label(
        value=label_text,
        layout=widgets.Layout(width='650px')
    )
    row_widget = widgets.HBox(
        [cb, label],
        layout=widgets.Layout(align_items='center', margin='0px', padding='0px')
    )

    cb.observe(make_observer(idx), names='value')
    display(row_widget)

In [14]:
len(to_remove)  # nombre d'outliers à supprimer


16

In [43]:
# finalisation du fichier nettoyé après validation manuelle

print(f"Points à supprimer : {sorted(to_remove)}")
df_clean = df.drop(index=list(to_remove)).reset_index(drop=True)

df_clean.to_csv("clean_calibration_sounds_features.csv", index=False)
print(f"Dataset : {len(df)} → {len(df_clean)} points après suppression")

Points à supprimer : [6, 14, 17, 24, 27, 54, 65, 71, 74, 104, 109, 118, 119, 131, 132]
Dataset : 143 → 128 points après suppression


Analyse complémentaires de distribution des vecteurs selon les dimensions les plus impactantes des models

In [44]:
import pandas as pd
import plotly.express as px

# Charger les données
df = pd.read_csv("calibration_sounds_features.csv", sep=",")

# Bornes des plages
bins = [0, 4, 8, 12, 16, 20]
labels = ["0-4mls", "4-8mls", "8-12mls", "12-16mls", "16-20mls"]

# Variable catégorielle
df["debit_range"] = pd.cut(
    df["debit"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Couleurs personnalisées (contrastées)
color_map = {
    "0-4mls": "blue",
    "4-8mls": "green",
    "8-12mls": "red",
    "12-16mls": "black",
    "16-20mls": "orange"   # 5e classe (si tu veux on peut aussi la garder rouge foncé)
}

# Scatter interactif
fig = px.scatter(
    df,
    x="mfcc12_mean",
    y="mfcc9_mean",
    color="debit_range",
    color_discrete_map=color_map,
    hover_data={
        "debit": True,
        "mfcc12_mean": True,
        "mfcc9_mean": True
    },
    title="Scatter plot MFCC12 vs MFCC9 par plage de débit"
)

fig.show()


In [45]:
df[["mfcc7_mean","spec_rolloff_mean", "debit"]].corr()

,mfcc7_mean,spec_rolloff_mean,debit
mfcc7_mean,1.000000,-0.202495,-0.272893
spec_rolloff_mean,-0.202495,1.000000,-0.270137
debit,-0.272893,-0.270137,1.000000


# détection importances des features

La permutation importance est le vrai critère de décision — une feature avec perm_mean <= 0 peut être supprimée sans perte de performance. Le test final compare le MAE CV avec et sans les features candidates à la suppression — si le MAE reste stable ou s'améliore, tu peux les retirer définitivement de extract_features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# ── 1. Charger le dataset ─────────────────────────────────────
calib = pd.read_csv("final_calibration_sounds_features.csv", sep=",")
X     = calib.iloc[:, :-1].values
y     = calib.iloc[:, -1].values
feature_names = calib.columns[:-1].tolist()

print(f"Dataset : {X.shape[0]} samples, {X.shape[1]} features")

# ── 2. Entraîner RandomForest ─────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

# ── 3. Feature importance intrinsèque (rapide) ────────────────
# Basée sur la réduction d'impureté moyenne dans les arbres
# Avantage : rapide
# Inconvénient : biaisée vers les features continues à forte cardinalité
importance_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\n=== Feature Importance intrinsèque (impureté) ===")
print(importance_df.to_string(index=False))

# ── 4. Permutation importance (plus fiable) ───────────────────
# Mesure la dégradation du MAE quand on mélange aléatoirement
# les valeurs d'une feature → si le MAE ne change pas, la feature
# est inutile. Plus lent mais plus honnête que l'importance intrinsèque.
perm = permutation_importance(
    rf, X, y,
    n_repeats  = 20,       # 20 permutations aléatoires par feature
    scoring    = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs     = -1
)

perm_df = pd.DataFrame({
    "feature"     : feature_names,
    "perm_mean"   : perm.importances_mean,
    "perm_std"    : perm.importances_std
}).sort_values("perm_mean", ascending=False).reset_index(drop=True)

print("\n=== Permutation Importance (plus fiable) ===")
print(perm_df.to_string(index=False))

# ── 5. Features à supprimer ───────────────────────────────────
# Une feature est candidate à la suppression si :
# - perm_mean <= 0  → supprimer la feature ne dégrade pas le MAE
# - perm_mean < perm_std  → importance non significative (bruit)
a_supprimer = perm_df[
    (perm_df["perm_mean"] <= 0) |
    (perm_df["perm_mean"] < perm_df["perm_std"])
]
print(f"\n⚠️  Features candidates à la suppression ({len(a_supprimer)}) :")
print(a_supprimer[["feature", "perm_mean", "perm_std"]].to_string(index=False))

# ── 6. Visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Importance intrinsèque
axes[0].barh(importance_df["feature"], importance_df["importance"])
axes[0].set_title("Feature Importance (impureté)")
axes[0].set_xlabel("Importance")
axes[0].invert_yaxis()

# Permutation importance
axes[1].barh(perm_df["feature"], perm_df["perm_mean"],
             xerr=perm_df["perm_std"], capsize=3)
axes[1].axvline(x=0, color="red", linestyle="--", label="seuil = 0")
axes[1].set_title("Permutation Importance")
axes[1].set_xlabel("Dégradation MAE si feature mélangée")
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

# ── 7. Test : MAE avec et sans les features candidates ────────
from sklearn.model_selection import cross_val_score

features_a_garder = perm_df[perm_df["perm_mean"] > perm_df["perm_std"]]["feature"].tolist()
idx_garder        = [feature_names.index(f) for f in features_a_garder]

X_reduit = X[:, idx_garder]

rf_reduit = RandomForestRegressor(n_estimators=200, random_state=42)
scores_complet = -cross_val_score(rf, X, y,
                                   cv=5, scoring="neg_mean_absolute_error")
scores_reduit  = -cross_val_score(rf_reduit, X_reduit, y,
                                   cv=5, scoring="neg_mean_absolute_error")

print(f"\n📊 MAE CV — toutes features ({X.shape[1]})     : {scores_complet.mean():.4f} ± {scores_complet.std():.4f}")
print(f"📊 MAE CV — features réduites ({X_reduit.shape[1]}) : {scores_reduit.mean():.4f} ± {scores_reduit.std():.4f}")
print(f"\n✅ Features conservées ({len(features_a_garder)}) :")
print(features_a_garder)

# dossier final 
  split des fichiers .wav retenus, création du dataset de features final, prêt pour l'entraînement des modèles de régression

In [46]:
import os
import re
import shutil
import pandas as pd

def extraire_debit_du_nom(filename):
    """
    Extrait la valeur de débit depuis le nom de fichier.
    Format attendu : (valeur)_mls_*.wav  ex: 5.675_mls_A1.wav
    Séparateur décimal : point uniquement.
    """
    match = re.match(r'^(\d+(?:\.\d+)?)_mls', filename)
    if not match:
        return None
    try:
        return round(float(match.group(1)), 3)
    except ValueError:
        return None


def copier_sans_outliers(df_outliers, dossier_source="sons", dossier_dest="cleaned_sounds"):

    debits_outliers = set(round(float(d), 3) for d in df_outliers["debit"])
    print(f"{len(debits_outliers)} débits outliers à exclure :")
    for d in sorted(debits_outliers):
        print(f"  {d:.3f} mL/s")

    os.makedirs(dossier_dest, exist_ok=True)

    fichiers = [f for f in os.listdir(dossier_source) if f.lower().endswith(".wav")]
    print(f"\n{len(fichiers)} fichiers .wav trouvés dans '{dossier_source}'")

    copies    = []
    exclus    = []
    non_match = []

    for f in sorted(fichiers):
        debit = extraire_debit_du_nom(f)

        if debit is None:
            print(f"  ⚠️  Nom non reconnu, copié par défaut : {f}")
            non_match.append(f)
            shutil.copy2(os.path.join(dossier_source, f),
                         os.path.join(dossier_dest, f))
            continue

        if debit in debits_outliers:
            exclus.append(f)
        else:
            shutil.copy2(os.path.join(dossier_source, f),
                         os.path.join(dossier_dest, f))
            copies.append(f)

    print(f"\n✅  {len(copies)} fichiers copiés")
    print(f"🗑️   {len(exclus)} fichiers exclus :")
    for f in exclus:
        print(f"     - {f}")
    if non_match:
        print(f"⚠️   {len(non_match)} fichiers au nom non parsable (copiés par défaut)")

    return copies, exclus




In [47]:
df_to_remove = df.loc[sorted(to_remove)]

In [48]:
df_to_remove

,mfcc1_mean,mfcc2_mean,mfcc3_mean,mfcc4_mean,mfcc5_mean,mfcc6_mean,mfcc7_mean,mfcc8_mean,mfcc9_mean,mfcc10_mean,...,rms_mean,rms_std,zero_cross_rate_mean,zero_cross_rate_std,flux_mean,flux_std,band_energy_mean,band_energy_std,debit,debit_range
6,-510.095856,58.360981,15.173298,17.274120,7.127695,9.113169,7.331812,7.145121,3.226136,5.423526,...,0.002199,0.000837,0.175425,0.159726,0.041671,0.018800,0.001324,0.000798,0.010,0-4mls
14,-266.870209,72.222000,-87.792702,-23.351536,-45.607628,-33.285030,9.173957,-25.980804,7.667041,2.699860,...,0.028688,0.013314,0.143383,0.047273,0.044582,0.023139,0.910318,0.710316,4.362,4-8mls
17,-261.016052,89.922066,-88.184547,-44.831287,-46.903713,-34.166832,11.806267,-16.663336,19.420057,1.729937,...,0.034329,0.012070,0.093225,0.017364,0.048622,0.018926,1.217915,0.636413,5.000,4-8mls
24,-262.917725,69.555687,-71.123482,-21.622292,-55.075966,-36.248009,4.902925,-25.395115,3.157986,-0.494796,...,0.031335,0.020705,0.139742,0.052933,0.046754,0.030020,0.724383,0.568205,5.777,4-8mls
27,-228.206131,76.782738,-79.285446,-40.655048,-48.811607,-37.400997,3.273486,-18.559147,10.256617,4.613340,...,0.040114,0.013291,0.106614,0.025524,0.045856,0.017005,1.273120,0.649136,6.250,4-8mls
54,-216.533325,69.501312,-72.617226,-34.720242,-57.414726,-36.951134,2.969132,-14.378023,10.249300,-0.372360,...,0.047220,0.017639,0.108126,0.027551,0.047900,0.021904,1.384519,0.674595,9.099,8-12mls
65,-178.671997,-12.057900,-43.653049,28.239906,-55.073109,14.799083,-19.792053,32.756882,2.223367,19.356733,...,0.043148,0.005620,0.381701,0.027875,0.023102,0.004101,0.779864,0.225033,10.660,8-12mls
71,-176.012222,27.351313,-18.944363,51.922634,-47.004757,10.187369,-27.266083,8.652864,-20.727484,0.092563,...,0.036810,0.003981,0.329691,0.027082,0.023517,0.003176,0.579802,0.132289,11.001,8-12mls
74,-165.691513,-13.326272,-44.091625,31.594826,-53.489311,12.249866,-25.373085,24.673876,0.893425,21.434336,...,0.049408,0.006906,0.386581,0.026557,0.023185,0.003843,0.880140,0.188028,11.110,8-12mls
104,-319.323029,46.175388,-7.094604,55.716812,-53.145176,13.111967,-23.448902,8.321729,-19.923458,2.405202,...,0.010163,0.001659,0.249338,0.044715,0.027176,0.005821,0.037722,0.021568,14.285,12-16mls


In [49]:
# ── Usage ─────────────────────────────────────────────────────────────────────

df_to_remove = df.loc[sorted(to_remove)]
copies, exclus = copier_sans_outliers(
    df_outliers   = df_to_remove,
    dossier_source= "calibration_sounds",
    dossier_dest  = "cleaned_sounds"
)

15 débits outliers à exclure :
  0.010 mL/s
  4.362 mL/s
  5.000 mL/s
  5.777 mL/s
  6.250 mL/s
  9.099 mL/s
  10.660 mL/s
  11.001 mL/s
  11.110 mL/s
  14.285 mL/s
  14.972 mL/s
  16.666 mL/s
  17.109 mL/s
  19.261 mL/s
  19.762 mL/s

143 fichiers .wav trouvés dans 'calibration_sounds'

✅  128 fichiers copiés
🗑️   15 fichiers exclus :
     - 0.01_mls.wav
     - 10.66_mls.wav
     - 11.001_mls.wav
     - 11.11_mls.wav
     - 14.285_mls_vox.wav
     - 14.972_mls_A51.wav
     - 16.666_mls_A50.wav
     - 17.109_mls_A51.wav
     - 19.261_mls_A51.wav
     - 19.762_mls_A50.wav
     - 4.362_mls_A51.wav
     - 5.777_mls_A51.wav
     - 5_mls_A50.wav
     - 6.25_mls_A50.wav
     - 9.099_mls_A50.wav


# Division des fichiers
fichers .wav retenus découpés en tranches de  xiemes secondes (ex 0,5)

In [50]:
import os
import shutil
import librosa
import soundfile as sf

def decouper_fichiers_wav(
    dossier_source      = 'cleaned_sounds',
    dossier_destination = 'final_calibration_sounds',
    duree_fenetre       = 0.5,
    sr_cible            = 32000
):
    # ── Nettoyage du dossier destination ─────────────────────────────
    if os.path.exists(dossier_destination):
        shutil.rmtree(dossier_destination)
        print(f"🗑️  Dossier '{dossier_destination}' vidé")
    os.makedirs(dossier_destination)
    print(f"📁  Dossier '{dossier_destination}' recréé")
    print("-" * 60)

    # ── Lecture des fichiers source ───────────────────────────────────
    fichiers = [f for f in os.listdir(dossier_source) if f.endswith('.wav')]

    if not fichiers:
        print("⚠️  Aucun fichier .wav trouvé dans", dossier_source)
        return

    n_samples_fenetre = int(duree_fenetre * sr_cible)
    total_segments    = 0

    print(f"📂 Source      : {dossier_source}  ({len(fichiers)} fichiers)")
    print(f"📂 Destination : {dossier_destination}")
    print(f"⏱️  Fenêtre     : {duree_fenetre}s  ({n_samples_fenetre} samples à {sr_cible} Hz)")
    print("-" * 60)

    for fichier in sorted(fichiers):
        chemin_source = os.path.join(dossier_source, fichier)
        nom_base      = os.path.splitext(fichier)[0]

        y, sr = librosa.load(chemin_source, sr=sr_cible, mono=True)

        n_segments = len(y) // n_samples_fenetre

        if n_segments == 0:
            print(f"⚠️  {fichier} trop court ({len(y)/sr:.2f}s) — ignoré")
            continue

        for i in range(n_segments):
            debut   = i * n_samples_fenetre
            fin     = debut + n_samples_fenetre
            segment = y[debut:fin]

            nom_segment   = f"{nom_base}_seg{i}.wav"
            chemin_sortie = os.path.join(dossier_destination, nom_segment)
            sf.write(chemin_sortie, segment, sr_cible, subtype='PCM_16')

        total_segments += n_segments
        print(f"✅  {fichier:<35}  →  {n_segments} segments "
              f"({len(y)/sr:.2f}s / {duree_fenetre}s)")

    print("-" * 60)
    print(f"✅  Total : {len(fichiers)} fichiers  →  {total_segments} segments")



In [51]:
# ── Appel de fonction ────────────────────────────────────────────────────────
decouper_fichiers_wav(
    dossier_source      = 'cleaned_sounds',
    dossier_destination = 'final_calibration_sounds',
    duree_fenetre       = 0.5,   # ← modifier ici
    sr_cible            = 32000
)

🗑️  Dossier 'final_calibration_sounds' vidé
📁  Dossier 'final_calibration_sounds' recréé
------------------------------------------------------------
📂 Source      : cleaned_sounds  (132 fichiers)
📂 Destination : final_calibration_sounds
⏱️  Fenêtre     : 0.5s  (16000 samples à 32000 Hz)
------------------------------------------------------------
✅  0.001_mls_A51.wav                    →  6 segments (3.00s / 0.5s)
✅  0.002_mls.wav                        →  6 segments (3.00s / 0.5s)
✅  0.003_mls.wav                        →  6 segments (3.00s / 0.5s)
✅  0.005_mls_iphone.wav                 →  6 segments (3.00s / 0.5s)
✅  0.006_mls_iphone_18db.wav            →  6 segments (3.00s / 0.5s)
✅  0.021_mls.wav                        →  6 segments (3.00s / 0.5s)
✅  0.02_mls_A51.wav                     →  6 segments (3.00s / 0.5s)
✅  0.1_mls_A51.wav                      →  6 segments (3.00s / 0.5s)
✅  0_mls_A51.wav                        →  6 segments (3.00s / 0.5s)
✅  10.394_mls_A51.wav        

# Nouvel embedding en features
Sur la base des fichiers .wav découpés, je réextrais les features pour créer un dataset de calibration plus riche et plus adapté à la prédiction en temps réel du débit urinaire.

In [52]:
build_dataset('final_calibration_sounds')

df=pd.read_csv("final_calibration_sounds_features.csv", sep=",")

Dataset saved as final_calibration_sounds_features.csv


données suplémentaires autres sources sonores